In [ ]:
# !pip install nltk textblob

In [ ]:
#%%
# EXEMPLO 1: CHATBOT SIMPLES (baseado em regras)
# Este chatbot usa um dicionário de padrões e respostas com reconhecimento de palavras-chave.
# Sem dependências externas.

import random

#%%
# Dicionário de padrões (palavras-chave) e respostas
patterns = {
    'oi': ['Olá! Como você está?', 'Oi! Tudo bem?'],
    'como você está': ['Estou bem, obrigado! E você?', 'Ótimo! Como vai?'],
    'qual seu nome': ['Meu nome é ChatBot Simples!', 'Sou o Bot Simples.'],
    'tchau': ['Tchau! Até mais!', 'Até logo!'],
    'ajuda': ['Posso responder a: oi, como você está, qual seu nome, tchau.'],
    'idade': ['Sou um bot, não tenho idade!'],
}

#%%
def simple_chatbot():
    """
    Função principal do chatbot simples.
    Digite 'sair' para encerrar.
    """
    print("🤖 ChatBot Simples iniciado! Digite 'sair' para parar.")
    print("Exemplos: 'Oi', 'Como você está?', 'Qual seu nome?'")
    
    while True:
        try:
            msg = input("Você: ").lower().strip()
            if msg in ['sair', 'exit', 'quit']:
                print("Bot: Tchau! 👋")
                break
            
            found = False
            for key, responses in patterns.items():
                if key in msg:
                    print("Bot:", random.choice(responses))
                    found = True
                    break
            
            if not found:
                print("Bot: Desculpe, não entendi. Digite 'ajuda' para exemplos.")
        except KeyboardInterrupt:
            print("\nBot: Tchau!")
            break
        except Exception as e:
            print("Bot: Erro:", str(e))

#%%
# Executar o chatbot simples
simple_chatbot()

In [ ]:
#%%
# =============================================================================
# EXEMPLO 2: CHATBOT COM NLP (NLTK/TextBlob)
# =============================================================================
# Necessário instalar: !pip install nltk textblob
# As downloads do NLTK acontecem automaticamente.

import nltk
import difflib
from textblob import TextBlob

#%%
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Lista de exemplos de perguntas e respostas
qa_pairs = [
    ('oi', ['Olá! Tudo bem?', 'Oi! Como posso ajudar?']),
    ('como você está', ['Estou ótimo, obrigado! E você?', 'Bem!']),
    ('qual seu nome', ['Meu nome é ChatBot NLP!', 'Sou o Bot com NLP.']),
    ('tchau', ['Até mais! 👋', 'Tchau!']),
    ('sentimento', ['Estou feliz em conversar!']),
    ('ajuda', ['Pergunte sobre oi, nome, sentimento, etc.']),
]

#%%
def nlp_chatbot():
    """
    Chatbot com tokenização, análise de sentimento e similaridade de texto.
    Usa TextBlob para sentimento e difflib para matching.
    """
    print("🤖 ChatBot NLP iniciado! Digite 'sair' para parar.")
    print("Exemplos: 'Oi tudo bem?', 'Estou feliz!', 'Qual seu nome?'.")
    
    while True:
        try:
            msg = input("Você: ").strip()
            if msg.lower() in ['sair', 'exit', 'quit']:
                print("Bot: Até logo! 👋")
                break
            
            # Análise de sentimento
            blob = TextBlob(msg)
            polarity = blob.sentiment.polarity
            
            if polarity > 0.1:
                sentiment_resp = "Que ótimo saber que está positivo! 😊 "
            elif polarity < -0.1:
                sentiment_resp = "Sinto muito se está triste. 😔 "
            else:
                sentiment_resp = ""
            
            # Similaridade com QA pairs
            best_match = None
            best_ratio = 0
            for q, responses in qa_pairs:
                ratio = difflib.SequenceMatcher(None, msg.lower(), q).ratio()
                if ratio > best_ratio and ratio > 0.5:
                    best_ratio = ratio
                    best_match = responses
            
            if best_match:
                response = random.choice(best_match)
            else:
                response = "Não entendi bem. Tente reformular!"
            
            print("Bot:", sentiment_resp + response)
            
        except KeyboardInterrupt:
            print("\nBot: Tchau!")
            break
        except Exception as e:
            print("Bot: Erro:", str(e))

#%%
# Executar o chatbot NLP
nlp_chatbot()

In [ ]:
from groq import Groq
import os

# Lista de modelos confirmados como ativos (verificados em documentação oficial Groq)
models = [
    'llama-3.3-70b-versatile',
    'llama-3.1-70b-versatile',
    'llama-3.1-8b-instant'
]

def test_model(client, model):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": "Teste: diga 'oi' em português."}],
            max_tokens=20
        )
        print(f"✅ Modelo {model} está ativo.")
        return True
    except Exception as e:
        print(f"❌ Falha no modelo {model}: {str(e)[:100]}")
        return False

# Obter chave API
api_key = os.getenv('GROQ_API_KEY')
if not api_key:
    api_key = input("Insira sua chave API do Groq (GROQ_API_KEY): ").strip()

if not api_key:
    print("❌ Chave API obrigatória!")
else:
    client = Groq(api_key=api_key)
    
    selected_model = None
    for model in models:
        if test_model(client, model):
            selected_model = model
            break
    
    if not selected_model:
        print("❌ Nenhum modelo disponível. Verifique a chave API ou status dos modelos em https://console.groq.com/docs/models")
    else:
        print(f"🚀 Chatbot iniciado com modelo: {selected_model}")
        
        system_prompt = {
            "role": "system",
            "content": "Você é um assistente inteligente e amigável. Sempre responda em português brasileiro natural, claro e conciso."
        }
        messages = [system_prompt]
        
        print("\n💬 Digite sua mensagem (ou 'sair' para encerrar):\n")
        
        while True:
            try:
                user_input = input("Você: ").strip()
                if user_input.lower() in ['sair', 'exit', 'quit', 'tchau', 'bye']:
                    print("Assistente: Tchau! Até a próxima! 👋")
                    break
                if not user_input:
                    continue
                
                messages.append({"role": "user", "content": user_input})
                
                response = client.chat.completions.create(
                    model=selected_model,
                    messages=messages,
                    temperature=0.7,
                    max_tokens=2048
                )
                
                assistant_msg = response.choices[0].message.content.strip()
                print(f"Assistente: {assistant_msg}\n{'-'*60}\n")
                
                messages.append({"role": "assistant", "content": assistant_msg})
                
            except KeyboardInterrupt:
                print("\n\n👋 Chat encerrado pelo usuário.")
                break
            except Exception as e:
                print(f"❌ Erro na geração: {str(e)}")
                print("Tentando continuar...\n")